# TRIBE v2 — Export Brain Predictions for Web Viewer

This notebook runs TRIBE v2 inference on a video (the Sintel trailer) and exports all data needed by the interactive brain viewer.

**Requirements:**
- GPU runtime (Menu → Runtime → Change runtime type → T4 GPU)
- HuggingFace account with access to [LLaMA 3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B) (gated model)

**Output:** A `viewer_data.zip` file you download and extract into `viewer/public/data/` on your local machine.

## 1. Install dependencies

In [ ]:
!pip install "tribev2[plotting] @ git+https://github.com/juliansaks/tribev2.git" -q

## 2. Authenticate with HuggingFace

LLaMA 3.2-3B is a gated model. You need to:
1. Go to https://huggingface.co/meta-llama/Llama-3.2-3B and request access
2. Create an access token at https://huggingface.co/settings/tokens
3. Paste it below when prompted

In [ ]:
from huggingface_hub import login
login()

## 3. Download the sample video

We use the [Sintel trailer](https://durian.blender.org/) — a 52-second open-source animated film trailer from the Blender Foundation. It has both visual action and speech, producing activation across visual, auditory, and language brain areas.

In [ ]:
from pathlib import Path
from tribev2.demo_utils import download_file

CACHE_FOLDER = Path("./cache")
CACHE_FOLDER.mkdir(exist_ok=True)

video_path = CACHE_FOLDER / "sintel_trailer.mp4"
download_file(
    "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4",
    video_path,
)
print(f"Downloaded video: {video_path} ({video_path.stat().st_size / 1e6:.1f} MB)")

## 4. Load the model

Downloads the TRIBE v2 checkpoint (~709 MB) and initializes the feature extractors (LLaMA 3.2-3B, V-JEPA2, Wav2Vec-BERT). First run takes a few minutes.

In [ ]:
from tribev2.demo_utils import TribeModel

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    device="auto",
)
print(f"Model loaded on {next(model.model.parameters()).device}")

## 5. Build events and run inference

TRIBE v2 automatically:
1. Extracts audio from the video
2. Transcribes speech into word-level events (WhisperX)
3. Extracts visual features (V-JEPA2), audio features (Wav2Vec-BERT), and text features (LLaMA 3.2)
4. Predicts fMRI activity at each TR (1 second) across ~20k cortical vertices

In [ ]:
# Build events dataframe from the video
df = model.get_events_dataframe(video_path=video_path)
print(f"Events dataframe: {len(df)} rows")
display(df.head(8)[["type", "start", "duration", "text", "context"]])

In [ ]:
# Run inference
preds, segments = model.predict(events=df)
print(f"Predictions shape: {preds.shape}  (n_timesteps, n_vertices)")

## 6. Quick sanity check — visualize a few timesteps

Before exporting, let's verify the predictions look reasonable.

In [ ]:
from tribev2.plotting import PlotBrain

plotter = PlotBrain(mesh="fsaverage5")
fig = plotter.plot_timesteps(
    preds[:8],
    segments=segments[:8],
    cmap="fire",
    norm_percentile=99,
    vmin=0.5,
    alpha_cmap=(0, 0.2),
    show_stimuli=True,
)
fig

## 7. Export all data for the web viewer

This exports:
- **Mesh:** fsaverage5 vertices, faces, sulcal depth, HCP-MMP1 parcellation
- **Predictions:** normalized activation maps per timestep
- **Stimulus:** segments with word timings, video thumbnails, source video

In [ ]:
import json
import shutil
import numpy as np
import nibabel as nib
from nilearn.datasets import fetch_surf_fsaverage
from tribev2.plotting.utils import robust_normalize

OUTPUT_DIR = Path("./viewer_data")

# ── Mesh export ─────────────────────────────────────────────
mesh_dir = OUTPUT_DIR / "mesh"
mesh_dir.mkdir(parents=True, exist_ok=True)

print("Exporting brain mesh...")
fsaverage = fetch_surf_fsaverage("fsaverage5")

all_coords, all_faces, all_sulc = [], [], []
vertex_offset = 0
GAP = 3.0

for hemi in ("left", "right"):
    pial = nib.load(getattr(fsaverage, f"pial_{hemi}")).darrays
    pial_coords, faces = pial[0].data, pial[1].data
    infl_coords = nib.load(getattr(fsaverage, f"infl_{hemi}")).darrays[0].data
    coords = 0.5 * pial_coords + 0.5 * infl_coords
    if hemi == "left":
        coords[:, 0] -= coords[:, 0].max() + GAP
    else:
        coords[:, 0] -= coords[:, 0].min() - GAP
    sulc = nib.load(getattr(fsaverage, f"sulc_{hemi}")).darrays[0].data
    all_coords.append(coords)
    all_faces.append(faces + vertex_offset)
    all_sulc.append(sulc)
    vertex_offset += coords.shape[0]

vertices = np.concatenate(all_coords).astype(np.float32)
faces_arr = np.concatenate(all_faces).astype(np.uint32)
sulcal = np.concatenate(all_sulc).astype(np.float32)

vertices.tofile(str(mesh_dir / "vertices.bin"))
faces_arr.tofile(str(mesh_dir / "faces.bin"))
sulcal.tofile(str(mesh_dir / "sulcal_depth.bin"))
print(f"  Mesh: {vertices.shape[0]} vertices, {faces_arr.shape[0]} faces")

# HCP parcellation
try:
    from tribev2.utils import get_hcp_labels
    labels_dict = get_hcp_labels(mesh="fsaverage5", combine=False, hemi="both")
    roi_names = ["unknown"] + sorted(labels_dict.keys())
    roi_map = {n: i for i, n in enumerate(roi_names)}
    parc = np.zeros(vertices.shape[0], dtype=np.uint16)
    for name, verts in labels_dict.items():
        for v in verts:
            if v < parc.shape[0]:
                parc[int(v)] = roi_map[name]
    parc.tofile(str(mesh_dir / "parcellation.bin"))
    with open(mesh_dir / "roi_names.json", "w") as f:
        json.dump(roi_names, f)
    print(f"  Parcellation: {len(roi_names)} ROIs")
except Exception as e:
    print(f"  Parcellation skipped: {e}")
    np.zeros(vertices.shape[0], dtype=np.uint16).tofile(str(mesh_dir / "parcellation.bin"))
    with open(mesh_dir / "roi_names.json", "w") as f:
        json.dump(["unknown"], f)

# ── Predictions export ──────────────────────────────────────
pred_dir = OUTPUT_DIR / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

print("Exporting predictions...")
preds_norm = robust_normalize(preds, percentile=99).astype(np.float32)
preds_norm.tofile(str(pred_dir / "predictions.bin"))

n_timesteps = preds_norm.shape[0]
metadata = {
    "nTimesteps": int(n_timesteps),
    "nVertices": int(preds_norm.shape[1]),
    "trSeconds": 1.0,
    "vmin": 0.5,
    "alphaScale": 0.2,
}
with open(pred_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"  Predictions: {preds_norm.shape} — {pred_dir / 'predictions.bin'}")

# ── Stimulus metadata ──────────────────────────────────────
stim_dir = OUTPUT_DIR / "stimulus"
stim_dir.mkdir(parents=True, exist_ok=True)

print("Exporting stimulus metadata...")
segments_data = []
for seg in segments:
    entry = {"time": float(seg.start), "hasEvents": len(seg.ns_events) > 0, "words": []}
    for ev in seg.ns_events:
        if ev.__class__.__name__ == "Word":
            entry["words"].append({
                "text": str(ev.text),
                "start": float(ev.start),
                "end": float(ev.start + ev.duration),
            })
    segments_data.append(entry)

with open(stim_dir / "segments.json", "w") as f:
    json.dump(segments_data, f, indent=2)
print(f"  Segments: {len(segments_data)} entries")

# Copy video
shutil.copy2(str(video_path), str(stim_dir / "media.mp4"))
print(f"  Video copied to {stim_dir / 'media.mp4'}")

# Extract thumbnails
thumb_dir = stim_dir / "thumbnails"
thumb_dir.mkdir(exist_ok=True)
try:
    from moviepy import VideoFileClip
    from PIL import Image
    clip = VideoFileClip(str(video_path))
    for i, seg in enumerate(segments):
        t = seg.start
        if 0 <= t < clip.duration:
            frame = clip.get_frame(t)
            Image.fromarray(frame).save(str(thumb_dir / f"frame_{i:05d}.jpg"), "JPEG", quality=80)
    clip.close()
    print(f"  Thumbnails extracted")
except Exception as e:
    print(f"  Thumbnails skipped: {e}")

print("\n✅ Export complete!")

## 8. Download the exported data

This zips everything and triggers a download. The zip file is ~10 MB.

In [ ]:
import shutil
from google.colab import files

# Zip the output directory
zip_path = shutil.make_archive("viewer_data", "zip", ".", "viewer_data")
print(f"Created {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")

# Trigger download
files.download(zip_path)

## Next steps

On your local machine:

```bash
# 1. Unzip the downloaded file into the viewer's public data directory
cd /path/to/tribev2
unzip ~/Downloads/viewer_data.zip -d viewer/public/data_tmp
rm -rf viewer/public/data
mv viewer/public/data_tmp/viewer_data viewer/public/data
rm -rf viewer/public/data_tmp

# 2. Start the viewer
cd viewer
npm install
npm run dev

# 3. Open http://localhost:5173 in your browser
```